In [1]:
import math
import time
import os
import pickle
from pyboolnet.external.bnet2primes import bnet_text2primes
from boolmore.mask import (
    generate_source_masks,
    mask_to_sources,
    mask_to_str,
)
from boolmore.phenotypes import (
    get_mintr_for_source_comb,
    find_matching_phenotype,
)

In [2]:
sources_partial = {"APC": 1}

# Choose a phenotype that (ideally) does not exist for worst-case benchmarking
target_phenotype = {"CD28": 0}

# Print a progress update every PRINT_INTERVAL seconds
PRINT_INTERVAL = 1.0

CACHE_FILE = "primes_cache.pkl"


In [3]:
with open(CACHE_FILE, "rb") as f:
    primes = pickle.load(f)
print("Loaded primes from cache.")

Loaded primes from cache.


In [4]:
source_nodes = [
    node
    for node in primes
    if primes[node] == [[{node: 0}], [{node: 1}]]
]
source_order = sorted(source_nodes)

n_free = len(source_order) - len(sources_partial)
n_total = 2 ** n_free

print(f"Source nodes      : {source_order}")
print(f"Partial assignment: {sources_partial}")
print(f"Target phenotype  : {target_phenotype}")
print(f"Free source nodes : {n_free}")
print(f"Total combinations: {n_total}")
print()

start = time.perf_counter()
last_print = start
processed = 0

for mask in generate_source_masks(source_order, sources_partial):

    processed += 1

    source_comb = mask_to_sources(mask, source_order)
    phenotypes = get_mintr_for_source_comb(primes, source_comb)

    match = find_matching_phenotype(
        {mask: phenotypes},
        target_phenotype,
    )

    if match is not None:
        elapsed = time.perf_counter() - start

        print("\nMATCH FOUND")
        print(f"Source combination: {mask_to_str(mask, len(source_order))}")
        print(f"Minimal trap space: {match[1]}")
        print(f"Processed          : {processed}/{n_total}")
        print(f"Elapsed            : {elapsed:.2f} s")
        break

    now = time.perf_counter()
    if now - last_print >= PRINT_INTERVAL:
        elapsed = now - start
        rate = processed / elapsed

        remaining = n_total - processed
        eta = remaining / rate if rate > 0 else math.inf
        total_est = n_total / rate if rate > 0 else math.inf

        print(
            f"{processed}/{n_total} "
            f"({100 * processed / n_total:.2f}%) | "
            f"{rate:.2f} comb/s | "
            f"ETA: {eta:.1f} s | "
            f"Estimated total: {total_est:.1f} s"
        )

        last_print = now

else:
    elapsed = time.perf_counter() - start

    print("\nNO MATCH FOUND")
    print(f"Processed: {processed}/{n_total}")
    print(f"Elapsed  : {elapsed:.2f} s")

Source nodes      : ['APC', 'DLL1', 'IFNA_e', 'IFNB_e', 'IFNG_e', 'IL10_e', 'IL12_e', 'IL15_e', 'IL18_e', 'IL1_e', 'IL21_e', 'IL23_e', 'IL25_e', 'IL27_e', 'IL29_e', 'IL2_e', 'IL33_e', 'IL36_e', 'IL4_e', 'IL6_e', 'IL7_e', 'TGFB_e']
Partial assignment: {'APC': 1}
Target phenotype  : {'CD28': 0}
Free source nodes : 21
Total combinations: 2097152

17/2097152 (0.00%) | 16.40 comb/s | ETA: 127868.8 s | Estimated total: 127869.8 s
36/2097152 (0.00%) | 17.61 comb/s | ETA: 119088.6 s | Estimated total: 119090.6 s
55/2097152 (0.00%) | 17.78 comb/s | ETA: 117914.4 s | Estimated total: 117917.5 s
76/2097152 (0.00%) | 18.47 comb/s | ETA: 113561.9 s | Estimated total: 113566.0 s
97/2097152 (0.00%) | 18.93 comb/s | ETA: 110779.8 s | Estimated total: 110784.9 s
119/2097152 (0.01%) | 19.40 comb/s | ETA: 108083.8 s | Estimated total: 108090.0 s
141/2097152 (0.01%) | 19.64 comb/s | ETA: 106761.3 s | Estimated total: 106768.5 s
161/2097152 (0.01%) | 19.62 comb/s | ETA: 106893.1 s | Estimated total: 106901

KeyboardInterrupt: 